> Part of **Complete Python Study Material** — split across per-chapter notebooks. See [`00_index.ipynb`](00_index.ipynb) for the notebook conventions, per-concept template, status tags, the digitalization log, chapter coverage tracker and cross-reference index.

## 15. Regular Expressions

*Scope:* Pattern matching over text.

### 15.1 Regex Fundamentals

A **regular expression** (**regex**) is a mini pattern-language — itself just a
string — that describes the *shape* of text rather than one exact piece of text. Where
`s == "A12"` only matches that one literal string, the pattern `[A-Z]\d{2}` describes an
entire family of strings: "one uppercase letter, then exactly two digits" — `"A12"`,
`"B07"`, `"Z99"`, and every other string with that shape, all at once.

**General applications** — the same small pattern language covers a surprising range of
everyday text-processing tasks:

| Application | What it looks like |
|---|---|
| Validation | does an input match an expected shape (email, phone number, product code)? |
| Searching / extraction | find every date, URL, or number buried in a block of text |
| Replacement / reformatting | rewrite matches (redact digits, normalize whitespace) |
| Splitting / tokenizing | break text apart on a pattern instead of one fixed delimiter |

The win over plain string methods is expressing a *shape* in one declarative pattern,
instead of several separate manual checks:

In [ ]:
import re

def is_valid_code_plain(s):   # plain string methods - two separate checks, both required
    return len(s) == 3 and s[0].isupper() and s[1:].isdigit()

def is_valid_code_regex(s):    # one pattern expresses the entire shape at once
    return bool(re.fullmatch(r"[A-Z]\d{2}", s))

print(is_valid_code_plain("A12"), is_valid_code_regex("A12"))   # True True
print(is_valid_code_plain("12A"), is_valid_code_regex("12A"))   # False False

### 15.2 The `re` Module

Python's regex support lives entirely in the built-in **`re`** module — there's no
special regex syntax in the language itself. A pattern is always just an ordinary
`str`, passed as an argument to a function like `re.search(pattern, text)`.

**Common mistake — forgetting raw strings.** Both Python string literals and regex
patterns use backslash escapes, and they don't agree on what every escape means. A
plain `"\d"` isn't a real Python escape sequence, so Python keeps it as the two
characters `\` and `d` — but warns about it, since that's rarely intentional. A raw
string (`r"..."`, 5.2.1) turns off Python's own escape processing entirely, so the
backslash reaches `re` exactly as typed. The convention is to **always** write regex
patterns as raw strings:

In [ ]:
plain_pattern = "\d{3}"   # SyntaxWarning: invalid escape sequence '\d' - works here, but by accident
raw_pattern = r"\d{3}"      # the correct, warning-free way to write any regex pattern

print(plain_pattern == raw_pattern)                 # True -> same two characters, in this particular case
print(bool(re.fullmatch(raw_pattern, "123")))   # True

**`re.compile(pattern)`** turns a pattern string into a reusable `Pattern` object.
`re.search(pattern, text)` compiles the same pattern internally on every single call;
compiling once with a pattern that's used repeatedly (in a loop, or across many calls)
avoids repeating that work, and reads cleanly since the pattern only has to be written
once. The resulting object exposes the same matching methods (`.match()`, `.search()`,
`.fullmatch()`, ...) without needing the pattern passed in again:

In [ ]:
digit_pattern = re.compile(r"\d+")   # compiled once - reusable

print(type(digit_pattern))                       # <class 're.Pattern'>
print(digit_pattern.match("42 apples"))     # <re.Match object; span=(0, 2), match='42'>
print(digit_pattern.search("apples: 42"))   # <re.Match object; span=(8, 10), match='42'>

**`.finditer(text)`** returns every non-overlapping match as a lazy iterator (14.1,
14.2) of `Match` objects, instead of collecting them all into a list up front —
`findall()` vs. `finditer()` is the same eager-vs-lazy trade-off as a list comprehension
vs. a generator expression (14.5). Each `Match` object carries `.group()` (the matched
text), `.start()`/`.end()` (its position), and `.span()` (both, as a tuple):

In [ ]:
text = "3 cats, 12 dogs, 7 birds"
matches = digit_pattern.finditer(text)   # a lazy iterator, nothing computed yet

print(type(matches))   # <class 'callable_iterator'>

for m in matches:
    print(m.group(), m.span())
# 3 (0, 1)
# 12 (8, 10)
# 7 (17, 18)

### 15.3 Character Classes

A **character class** — `[...]` — matches exactly **one** character, chosen from
whatever is listed inside the brackets. A range like `a-z` or `0-9` can stand in for
listing every character individually, and ranges can be combined in one class:

In [ ]:
text = "The Quick Brown Fox"
print(re.findall(r"[aeiouAEIOU]", text))   # every vowel, one character per match

color = "#1a2B3c"
print(bool(re.fullmatch(r"#[0-9a-fA-F]{6}", color)))   # True -> a valid 6-digit hex color

Putting `^` **first** inside the brackets negates the class — it then matches any
character *not* in the set:

In [ ]:
phone = "call (555) 123-4567 now"
digits_only = re.sub(r"[^0-9]", "", phone)   # replace every character that's NOT a digit with ""
print(digits_only)   # 5551234567

### 15.4 Predefined Character Classes

Some character classes come up so often that `re` provides a one-character shorthand
for each — no brackets needed:

| Shorthand | Equivalent to | Matches |
|---|---|---|
| `\d` | `[0-9]` | a digit |
| `\D` | `[^0-9]` | a non-digit |
| `\w` | `[a-zA-Z0-9_]` | a "word" character (letter, digit, or underscore) |
| `\W` | `[^a-zA-Z0-9_]` | a non-word character |
| `\s` | `[ \t\n\r\f\v]` | a whitespace character |
| `\S` | `[^ \t\n\r\f\v]` | a non-whitespace character |
| `.` | — | any character except a newline |

Each is a drop-in replacement anywhere a character class could go, and combines with
quantifiers (15.5) exactly the same way:

In [ ]:
text = "Order #482: 3 items, ship by 2026-08-24"

print(re.findall(r"\d+", text))   # ['482', '3', '2026', '08', '24'] -> runs of digits
print(re.findall(r"\w+", text))   # ['Order', '482', '3', 'items', 'ship', 'by', '2026', '08', '24']
print(re.split(r"\s+", text))       # split on any run of whitespace
print(re.findall(r"\D+", text))   # runs of NON-digit characters, punctuation and spaces included

**Common mistake — treating `\b` as a character class.** `\b` (word boundary) looks
like it belongs on this list, but it's different in kind: it matches a *position*
(between a `\w` character and a non-`\w` character), not an actual character, so it
never consumes anything itself. Without it, a plain search for `"cat"` matches inside
unrelated words too:

In [ ]:
text = "cat catalog concatenate"
print(re.findall(r"cat", text))          # ['cat', 'cat', 'cat'] -> matches inside other words too
print(re.findall(r"\bcat\b", text))   # ['cat'] -> only the whole word

### 15.5 Quantifiers

A **quantifier** attaches to whatever comes right before it (a literal character, a
character class, or a group) and says *how many times* to repeat it:

| Quantifier | Means |
|---|---|
| `*` | 0 or more |
| `+` | 1 or more |
| `?` | 0 or 1 (optional) |
| `{n}` | exactly `n` |
| `{n,}` | `n` or more |
| `{n,m}` | between `n` and `m` (inclusive) |

In [ ]:
print(re.findall(r"ab*", "a ab abb abbb"))       # * -> 0+ 'b': ['a', 'ab', 'abb', 'abbb']
print(re.findall(r"ab+", "a ab abb abbb"))       # + -> 1+ 'b': ['ab', 'abb', 'abbb'] ("a" alone drops out)
print(re.findall(r"colou?r", "color colour"))   # ? -> optional 'u': ['color', 'colour']
print(re.findall(r"\d{4}", "2026-08-24"))         # {4} -> exactly 4 digits: ['2026']
print(re.findall(r"\d{2,4}", "12 123 1234"))     # {2,4} -> between 2 and 4 digits

**Greedy vs. lazy.** Every quantifier above is **greedy** by default — it matches as
*much* text as it possibly can. Appending `?` to a quantifier (`*?`, `+?`, `??`) makes
it **lazy** instead — it matches as *little* as it can get away with. The difference is
easy to miss until it silently swallows far more than intended:

In [ ]:
html = "<b>bold</b> and <i>italic</i>"

print(re.findall(r"<.*>", html))     # greedy - .* grabs as much as possible, spans BOTH tags
print(re.findall(r"<.*?>", html))   # lazy - the trailing ? stops at the FIRST '>' instead

### 15.6 The Full `re` Module Function Catalog

Every top-level function `re` provides, in one place — `compile()` and `finditer()`
were already covered in depth in 15.2:

| Function | Does |
|---|---|
| `re.match(pattern, s)` | matches only at the **start** of `s` |
| `re.fullmatch(pattern, s)` | the **entire** string must match |
| `re.search(pattern, s)` | finds the **first** match anywhere in `s` |
| `re.findall(pattern, s)` | returns **every** match as a `list` of strings |
| `re.finditer(pattern, s)` | returns every match as a lazy iterator of `Match` objects (15.2) |
| `re.sub(pattern, repl, s)` | returns `s` with every match replaced by `repl` |
| `re.subn(pattern, repl, s)` | same as `sub()`, plus the number of replacements made |
| `re.split(pattern, s)` | splits `s` wherever `pattern` matches |
| `re.compile(pattern)` | precompiles into a reusable `Pattern` object (15.2) |
| `re.escape(s)` | escapes every regex-special character in `s`, for safe use as a literal pattern |

`match`/`fullmatch`/`search`/`findall` side by side, on the same input, showing exactly
how each one differs:

In [ ]:
print(re.match(r"\d+", "42 apples"))        # <re.Match object; span=(0, 2), match='42'>
print(re.match(r"\d+", "apples 42"))        # None -> doesn't start with a digit

print(re.fullmatch(r"\d+", "42"))              # <re.Match object; span=(0, 2), match='42'>
print(re.fullmatch(r"\d+", "42 apples"))    # None -> trailing text breaks a full match

print(re.search(r"\d+", "apples 42"))        # <re.Match object; span=(7, 9), match='42'>

print(re.findall(r"\d+", "3 cats, 12 dogs"))   # ['3', '12']

`sub`/`subn`/`split`/`escape`, each with a minimal example:

In [ ]:
print(re.sub(r"\d+", "#", "3 cats, 12 dogs"))     # # cats, # dogs -> every match replaced
print(re.subn(r"\d+", "#", "3 cats, 12 dogs"))     # ('# cats, # dogs', 2) -> plus a replacement count

print(re.split(r"\s*,\s*", "red, green,blue ,  yellow"))   # ['red', 'green', 'blue', 'yellow']

user_input = "3.14 (approx.)"
print(re.escape(user_input))                                          # every special char escaped
print(bool(re.fullmatch(re.escape(user_input), user_input)))   # True -> now safe as a literal pattern

In [ ]:
# --- 15. Regular Expressions — scratch cell ---
# Experiments for this chapter. Promote anything worth keeping into the
# relevant section as a proper example cell.
